In [6]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import torchvision
from torchvision import datasets, transforms
from torchvision.datasets import MovingMNIST

torch.manual_seed(0)
np.random.seed(0)

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
print("torch", torch.__version__, "| device:", device)

torch 2.11.0+cpu | device: cpu


In [ ]:
# ── MNIST, binarised, flattened to 784-vectors ────────────────────────────────
# TAKEN FROM LECTURE 11 NOTEBOOK -- ONLY FOR FLATTENED, NONMOVING IMAGES
tf = transforms.ToTensor()
train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=tf)

def to_tensors(ds, n=None):
    X = ds.data.float() / 255.0
    Y = ds.targets.clone()
    if n is not None:
        g = torch.Generator().manual_seed(0)
        sel = torch.randperm(len(X), generator=g)[:n]
        X, Y = X[sel], Y[sel]
    return (X.reshape(len(X), -1) > 0.5).float(), Y

X_tr, Y_tr = to_tensors(train_ds, n=20000)
X_te, Y_te = to_tensors(test_ds,  n=4000)
X_tr, Y_tr = X_tr.to(device), Y_tr.to(device)
X_te, Y_te = X_te.to(device), Y_te.to(device)

D, C = X_tr.shape[1], 10          # 784 pixels, 10 classes
print("train", tuple(X_tr.shape), " test", tuple(X_te.shape))

def onehot(y):
    return F.one_hot(y, C).float()

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.18MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 135kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.29MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.79MB/s]


train (20000, 784)  test (4000, 784)


In [ ]:
# MOVING MNIST FRAMES
full = MovingMNIST(root="./data", split=None, download=True)
print(len(full), full[0].shape)   # expect 10000, torch.Size([20, 1, 64, 64])

X = torch.stack([full[i] for i in range(len(full))]).float().div(255).squeeze(2)  # (10000, 20, 64, 64)

X_tr, X_te = X[:8000], X[8000:]
ctx_tr, fut_tr = X_tr[:, :5], X_tr[:, 5:]    # 5 context frames, 15 future
ctx_te, fut_te = X_te[:, :5], X_te[:, 5:]
print(X_tr.shape, X_te.shape)

# should output the tensor sizes for hte images
# 8000 training, 2000 testing, 20 frames for each image

10000 torch.Size([20, 1, 64, 64])
torch.Size([8000, 20, 64, 64]) torch.Size([2000, 20, 64, 64])
